# SCAG Demo — M&Ms-2 LAX-4C

Step-by-step Semantic Coactivation Graph analysis for **M&Ms-2 LAX-4C**.

Input policy: **ED and ES LAX-4C images per patient**.

The notebook exposes the full analysis rather than hiding it inside `run_model()`:

**data → trained model → activations → X → R → Q → R–Q alignment → graph → communities → semantic analyses → causal ablation**


## 1. Imports

In [ ]:
import numpy as np
import torch
from collections import Counter
from IPython.display import display

from scag.config import SCAGConfig
from scag.data import (
    load_datasets,
    build_graph_analysis_cohort,
    make_split_safe,
    build_loaders,
    phase_encode_segments,
)
from scag.models import build_model
from scag.activations import extract_activations
from scag.semantics import compute_neuron_concept_matrix
from scag.matrices import compute_R, compute_Q, align_R_Q, compute_edge_reliability
from scag.graph import (
    analyze_graph,
    density_threshold,
    seed_stability,
    community_cards,
    area_coupling,
)
from scag.ablation import community_ablation
from scag.pipeline import compute_area_profiles, run_semantic_granularity

## 2. Configuration

In [ ]:
cfg = SCAGConfig(dataset="mnms2")

print("Dataset:", cfg.dataset)
print("Device:", cfg.device)
print("Data root:", cfg.data_root)
print("Model root:", cfg.model_root)
print("Classes:", cfg.classes)
print("Concepts:", cfg.concept_names)

The notebook uses the full SCAG defaults. For a quick code smoke test only, you can temporarily lower the permutation/ablation counts in `cfg`; use the defaults for reported experiments.

## 3. Load the official train / validation / test splits

In [ ]:
train_dataset, val_dataset, test_dataset, splits = load_datasets(cfg)

print("Train:", len(train_dataset))
print("Validation:", len(val_dataset))
print("Test:", len(test_dataset))

## 4. Build the graph-analysis cohort

The graph-analysis cohort is **official validation + official test**, matching the existing SCAG experiments. Therefore the official test split is not treated as an untouched test set for this analysis.

In [ ]:
eval_dataset = build_graph_analysis_cohort(
    val_dataset,
    test_dataset,
    cfg,
)
eval_dataset = make_split_safe(
    train_dataset,
    eval_dataset,
)

print("Graph-analysis cohort:", len(eval_dataset))

## 5. Build deterministic loaders

In [ ]:
train_loader, eval_loader = build_loaders(
    train_dataset,
    eval_dataset,
    batch_size=cfg.batch_size,
    num_workers=cfg.num_workers,
)

print("Train batches:", len(train_loader))
print("Evaluation batches:", len(eval_loader))

## 6. Load one trained architecture

In [ ]:
tag = "resnet18"

model, hook_target, layer_name = build_model(tag, cfg)
model = model.to(cfg.device).eval()

print("Architecture:", tag)
print("Feature layer:", layer_name)

## 7. Extract training activations and masks

In [ ]:
train_acts, train_segs = extract_activations(
    model,
    train_loader,
    hook_target,
    cfg.device,
)

print("Train activations:", tuple(train_acts.shape))
print("Train masks:", tuple(train_segs.shape))

## 8. Extract graph-cohort activations and masks

In [ ]:
eval_acts, eval_segs = extract_activations(
    model,
    eval_loader,
    hook_target,
    cfg.device,
)

print("Evaluation activations:", tuple(eval_acts.shape))
print("Evaluation masks:", tuple(eval_segs.shape))

## 9. Convert anatomy masks to phase-aware concepts

The shared semantic vocabulary is:

- `RV_ED`, `MYO_ED`, `LV_ED`
- `RV_ES`, `MYO_ES`, `LV_ES`

with background kept as concept `0`.

In [ ]:
train_segs = phase_encode_segments(train_segs)
eval_segs = phase_encode_segments(eval_segs)

print("Encoded labels:", torch.unique(train_segs).tolist())

## 10. Compute the neuron–concept association matrix X

`X[k, c]` measures the association between neuron/filter `k` and semantic concept `c` using the SCAG dissection procedure.

In [ ]:
X, labeled = compute_neuron_concept_matrix(
    train_acts,
    train_segs,
    num_concepts=cfg.num_concepts,
    alpha=cfg.alpha,
    eta=cfg.eta,
)

lab_idx = np.where(labeled)[0]

print("X shape:", X.shape)
print("Labeled neurons:", len(lab_idx), "/", X.shape[0])

## 11. Inspect dominant concepts

In [ ]:
concept_counts = Counter(
    X[lab_idx].argmax(axis=1).tolist()
)

{
    cfg.concept_name_map[int(concept_id)]: int(count)
    for concept_id, count in sorted(concept_counts.items())
}

## 12. Compute functional coactivation R

Each activation map is spatially pooled, then neuron–neuron Spearman correlation is computed.

In [ ]:
R_train = compute_R(train_acts)
R_eval = compute_R(eval_acts)

print("R_train:", R_train.shape)
print("R_eval:", R_eval.shape)

## 13. Compute semantic similarity Q

`Q` is cosine similarity between the semantic concept profiles of the **same labeled neurons**.

In [ ]:
Q = compute_Q(X[lab_idx])

print("Q:", Q.shape)

## 14. Train–evaluation edge-weight reproducibility

In [ ]:
edge_reliability = compute_edge_reliability(
    R_train,
    R_eval,
    lab_idx,
)

print(f"Spearman rho = {edge_reliability:.4f}")

## 15. Full-matrix R–Q alignment

In [ ]:
R_train_lab = R_train[np.ix_(lab_idx, lab_idx)]
R_eval_lab = R_eval[np.ix_(lab_idx, lab_idx)]

alignment_train = align_R_Q(
    R_train_lab,
    Q,
    n_perm=cfg.n_mantel_permutations,
)

alignment_eval = align_R_Q(
    R_eval_lab,
    Q,
    n_perm=cfg.n_mantel_permutations,
)

print("Train (r, z, p):", alignment_train)
print("Eval  (r, z, p):", alignment_eval)

## 16. Analyze the fixed-threshold graph

In [ ]:
graph_eval, artifacts_eval = analyze_graph(
    R_eval,
    X,
    labeled,
    tau=cfg.tau,
    n_perm=cfg.n_battery_permutations,
)

graph_eval

## 17. Inspect Louvain communities

In [ ]:
communities = artifacts_eval["communities"]

print("Number of communities:", len(communities))
for community_id, community in enumerate(communities):
    print(f"Community {community_id:02d}: {len(community)} neurons")

## 18. Analyze a density-matched graph

In [ ]:
tau_dm = density_threshold(
    R_eval,
    cfg.graph_density,
)

graph_dm, artifacts_dm = analyze_graph(
    R_eval,
    X,
    labeled,
    tau=tau_dm,
    n_perm=cfg.n_battery_permutations,
)

print("Density-matched threshold:", tau_dm)
print("Edges:", graph_dm["n_edges"])
print("Modularity:", graph_dm["modularity"])

## 19. Compare semantic granularity

The **same neurons, R matrix, edge weights, and thresholds** are interpreted at three semantic levels:

1. anatomy,
2. cardiac phase,
3. anatomy × phase.

In [ ]:
granularity_df, granularity_summary, semantic_views = run_semantic_granularity(
    tag=tag,
    R_train=R_train,
    R_eval=R_eval,
    X=X,
    labeled=labeled,
    tau_dm=tau_dm,
    cfg=cfg,
)

display(granularity_df)

## 20. Louvain seed stability

In [ ]:
ami_mean, ami_sd, partition_stability = seed_stability(
    artifacts_eval["G"],
    artifacts_eval["lab_nodes"],
    artifacts_eval["labels"],
    n_seeds=cfg.n_louvain_seeds,
)

print("AMI vs semantics:", ami_mean, "+/-", ami_sd)
print("Partition self-stability:", partition_stability)

## 21. Community semantic enrichment

In [ ]:
enrichment_df = community_cards(
    artifacts_eval,
    cfg.concept_name_map,
    cfg.concept_category_map,
)

display(enrichment_df)

## 22. Anatomy-area coupling

In [ ]:
area_profiles = compute_area_profiles(
    train_segs,
    cfg.num_concepts,
)

area_stats = area_coupling(
    R_eval,
    X,
    labeled,
    area_profiles,
    n_perm=cfg.n_cooccurrence_permutations,
)

area_stats

## 23. Causal community ablation

Each Louvain community is ablated by zeroing its feature channels at the selected layer. Its prediction effect is compared with size-matched random-neuron ablations.

**Note:** the default `n_ablation_samples=1000` is intentionally expensive for final experiments.

In [ ]:
node2label = dict(zip(
    artifacts_eval["lab_nodes"],
    artifacts_eval["labels"].tolist(),
))

ablation_df = community_ablation(
    model=model,
    hook_target=hook_target,
    loader=eval_loader,
    communities=artifacts_eval["communities"],
    num_filters=train_acts.shape[1],
    device=cfg.device,
    val_segs=eval_segs,
    node2label=node2label,
    concept_names=cfg.concept_name_map,
    n_random=cfg.n_ablation_samples,
)

display(ablation_df)

## 24. Rank communities by causal effect

In [ ]:
if ablation_df is not None and len(ablation_df):
    columns = [
        column
        for column in ["comm", "size", "kl", "rnd", "rnd_sd", "flip", "specificity", "spec_ratio"]
        if column in ablation_df.columns
    ]
    display(
        ablation_df[columns]
        .sort_values("kl", ascending=False)
        .reset_index(drop=True)
    )

## Outputs produced in this notebook

- `X`: neuron–concept association matrix
- `R_train`, `R_eval`: functional coactivation matrices
- `Q`: neuron–neuron semantic similarity matrix
- `alignment_train`, `alignment_eval`: R–Q alignment
- `artifacts_eval["G"]`: coactivation graph
- `artifacts_eval["communities"]`: Louvain communities
- `granularity_df`: anatomy vs phase vs anatomy×phase
- `enrichment_df`: community semantic enrichment
- `area_stats`: anatomical-burden coupling
- `ablation_df`: causal community-ablation results

For all configured architectures, run:

```bash
python run_scag.py --dataset mnms2
```
